# Notebook 07 — Marker Genes and Cell Type Annotation

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 07 of 12  
**Time estimate:** 75 minutes

> Leiden clustering gives you numbered clusters. Marker genes tell you what
> those clusters actually are biologically.
> The statistical test is Wilcoxon; the annotation is biology.

---
## Step 1 — Motivation

After clustering, "Cluster 2" has no biological meaning without knowing which genes
distinguish it from other clusters. Marker gene analysis identifies the genes most
specifically expressed in each cluster, allowing researchers to annotate clusters
as known cell types (e.g. "Cluster 2 = CD14+ Monocytes").

---
## Step 2 — Intuition

**What makes a gene a marker?**  
Two criteria: (1) it is expressed significantly higher in one cluster vs. others
(statistical significance), and (2) the fold-change is large enough to be biologically
meaningful (effect size). We use the Wilcoxon rank-sum test for (1) and log2 fold
change for (2).

**One-vs-rest strategy:** For each cluster, test each gene comparing that cluster's
cells against all other cells combined. This gives a list of markers per cluster.

**Annotation:** The researcher then looks up the top markers in known cell-type
databases (CellMarker, PanglaoDB, Human Cell Atlas) to assign a biological label.

---
## Step 3 — Biological Background

**Classic PBMC markers (T-cell panel):**
- CD3E, CD3D, CD3G — pan T-cell markers
- CD4 — CD4+ T helper cells
- CD8A, CD8B — CD8+ cytotoxic T cells
- FOXP3, IL2RA (CD25) — Regulatory T cells (Tregs)
- MS4A1 (CD20), CD79A — B cells
- CD14, LYZ, S100A8 — CD14+ Monocytes
- GNLY, NKG7 — NK cells
- PPBP — Platelets/Megakaryocytes

**Why Wilcoxon, not t-test?**  
scRNA-seq data is non-normal (heavy-tailed, zero-inflated). The Wilcoxon rank-sum test
is non-parametric — it makes no distributional assumption. It also has good power
for detecting differences in skewed distributions. The t-test assumes normality and
can inflate false positives with zero-inflated data.

---
## Step 4 — Mathematical Explanation

**Wilcoxon rank-sum test (Mann-Whitney U):**  
Given expression values of gene $g$ in cluster $c$ ($x_1, \ldots, x_{n_c}$) vs.
rest ($y_1, \ldots, y_{n_r}$), rank all $n_c + n_r$ values together.

$$U = \sum_{i=1}^{n_c} \sum_{j=1}^{n_r} \mathbf{1}[x_i > y_j]$$

Under $H_0$: $\mathbb{E}[U] = n_c n_r / 2$,
$\text{Var}(U) = n_c n_r (n_c + n_r + 1) / 12$

$$z = \frac{U - \mathbb{E}[U]}{\sqrt{\text{Var}(U)}}$$

The p-value is $2 \cdot (1 - \Phi(|z|))$ (two-sided), or one-sided if testing for
upregulation only. After computing p-values for all genes, apply BH correction (NB12).

**Log2 fold change:**
$$\text{log2FC} = \log_2\left(\frac{\text{mean}(x + 1)}{\text{mean}(y + 1)}\right)$$

Markers are typically filtered to: adjusted p-value < 0.05 AND |log2FC| > 0.5 (i.e.
at least 1.4-fold difference).

In [ ]:
# Step 6 — Python: Wilcoxon marker gene test from scratch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ---- Simulate log-normalized scRNA-seq data with known markers ----
N_CELLS = 300
N_GENES = 500
cell_types = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS, p=[0.4, 0.35, 0.25])
gene_names = [f'Gene{i:03d}' for i in range(N_GENES)]
# First 20 genes per cell type are 'real' markers
MARKERS = {
    'T_cell':   list(range(0, 20)),
    'B_cell':   list(range(20, 40)),
    'Monocyte': list(range(40, 60)),
}

def sim_scrna(n_cells, n_genes, cell_types, markers, rng):
    X = rng.lognormal(0, 0.5, (n_cells, n_genes)) - 0.8
    X = np.clip(X, 0, None)
    # Zero out ~70% (sparsity)
    X[rng.random(X.shape) < 0.70] = 0
    for i, ct in enumerate(cell_types):
        X[i, markers[ct]] += rng.exponential(2.5, len(markers[ct]))
    return X

X = sim_scrna(N_CELLS, N_GENES, cell_types, MARKERS, rng)
# Use true cell types as cluster labels for this notebook
cluster_labels = np.array(cell_types)
uniq_clusters = sorted(set(cell_types))

# ---- Wilcoxon rank-sum test (one-vs-rest) ----
def wilcoxon_markers(X, cluster_labels, gene_names, min_log2fc=0.25):
    """Find marker genes for each cluster via one-vs-rest Wilcoxon test."""
    uniq = sorted(set(cluster_labels))
    results = {}
    for cl in uniq:
        mask_in  = cluster_labels == cl
        mask_out = ~mask_in
        X_in  = X[mask_in, :]
        X_out = X[mask_out, :]

        n_genes = X.shape[1]
        pvals   = np.ones(n_genes)
        log2fcs = np.zeros(n_genes)

        for g in range(n_genes):
            a = X_in[:, g]
            b = X_out[:, g]
            # Wilcoxon (Mann-Whitney U)
            stat, p = stats.mannwhitneyu(a, b, alternative='greater')
            pvals[g] = p
            mean_in  = a.mean() + 1e-9
            mean_out = b.mean() + 1e-9
            log2fcs[g] = np.log2(mean_in / mean_out)

        # BH correction
        m = n_genes
        rank = np.argsort(pvals)
        adj_pvals = np.ones(m)
        for i, r in enumerate(rank):
            adj_pvals[r] = min(pvals[r] * m / (i + 1), 1.0)
        # Enforce monotonicity
        min_so_far = 1.0
        for r in rank[::-1]:
            adj_pvals[r] = min(adj_pvals[r], min_so_far)
            min_so_far = adj_pvals[r]

        # Filter: significant + min log2FC
        sig = (adj_pvals < 0.05) & (log2fcs > min_log2fc)
        marker_genes = np.where(sig)[0]
        order = np.argsort(pvals[marker_genes])
        results[cl] = {
            'genes': [gene_names[g] for g in marker_genes[order]],
            'log2fc': log2fcs[marker_genes[order]],
            'adj_pval': adj_pvals[marker_genes[order]],
            'indices': marker_genes[order],
        }
    return results

marker_results = wilcoxon_markers(X, cluster_labels, gene_names)

for cl, res in marker_results.items():
    n_found = len(res['genes'])
    true_markers = set(MARKERS[cl])
    found_idx = set(res['indices'].tolist())
    tp = len(true_markers & found_idx)
    print(f'Cluster {cl}: {n_found} markers found, {tp}/20 true markers recovered')
    print(f'  Top 5: {res["genes"][:5]}')

In [ ]:
# Step 7 — Visualization: Dotplot and volcano plot of markers
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Volcano-style plot for T_cell markers
ax = axes[0]
cl = 'T_cell'
mask_in  = cluster_labels == cl
mask_out = ~mask_in
all_log2fc = np.zeros(N_GENES)
all_adjp   = np.ones(N_GENES)
for g in range(N_GENES):
    a, b = X[mask_in, g], X[mask_out, g]
    _, p = stats.mannwhitneyu(a, b, alternative='greater')
    all_log2fc[g] = np.log2((a.mean()+1e-9) / (b.mean()+1e-9))
    all_adjp[g] = p
# Re-use BH from above (approx)
m = N_GENES
rank = np.argsort(all_adjp)
adj = np.ones(m)
for i, r in enumerate(rank):
    adj[r] = min(all_adjp[r] * m / (i+1), 1.0)
min_sf = 1.0
for r in rank[::-1]:
    adj[r] = min(adj[r], min_sf)
    min_sf = adj[r]

neg_log10_p = -np.log10(adj + 1e-300)
colors_vol = np.where(
    (adj < 0.05) & (all_log2fc > 0.5), 'tomato',
    np.where((adj < 0.05) & (all_log2fc < -0.5), 'steelblue', 'lightgrey')
)
ax.scatter(all_log2fc, neg_log10_p, c=colors_vol, s=8, alpha=0.6)
ax.axhline(-np.log10(0.05), color='black', ls='--', lw=0.8)
ax.axvline(0.5, color='grey', ls='--', lw=0.8)
ax.axvline(-0.5, color='grey', ls='--', lw=0.8)
ax.set_xlabel('log2 Fold Change')
ax.set_ylabel('-log10(adjusted p-value)')
ax.set_title(f'A. Volcano plot — T_cell markers\n(red = upregulated; blue = downregulated)')
# Annotate top 3
top3 = np.argsort(neg_log10_p)[::-1][:3]
for idx in top3:
    ax.annotate(gene_names[idx], (all_log2fc[idx], neg_log10_p[idx]),
                 fontsize=7, alpha=0.8)

# Panel B: Heatmap of top 5 markers per cluster
ax = axes[1]
top5_per_cluster = []
top5_names = []
for cl in uniq_clusters:
    res = marker_results[cl]
    top5_per_cluster.append(res['indices'][:5])
    top5_names += res['genes'][:5]
all_top_idx = np.concatenate(top5_per_cluster)
if len(all_top_idx) > 0:
    X_top = X[:, all_top_idx]
    # Sort cells by cluster
    sort_order = np.argsort([uniq_clusters.index(ct) for ct in cell_types])
    im = ax.imshow(X_top[sort_order, :].T, aspect='auto', cmap='YlOrRd',
                    vmin=0, vmax=np.percentile(X_top, 95), interpolation='none')
    plt.colorbar(im, ax=ax, label='Expression')
    ax.set_yticks(range(len(top5_names)))
    ax.set_yticklabels(top5_names, fontsize=6)
    ax.set_xlabel('Cells (sorted by cluster)')
    ax.set_title('B. Top 5 marker genes per cluster\n(heatmap)')
    # Add cluster boundaries
    ct_sorted = np.array(cell_types)[sort_order]
    boundaries = np.where(ct_sorted[:-1] != ct_sorted[1:])[0]
    for b in boundaries:
        ax.axvline(b + 0.5, color='blue', lw=0.8)

plt.suptitle('Module 10 NB07: Marker Genes and Cell Type Annotation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_markers.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `wilcoxon_one_vs_rest(X, labels, cluster_id)` returning a ranked list
   of marker genes.
2. Change `min_log2fc` to 1.0. How many markers are recovered for each cluster?
3. Look up CD3E in the literature. Which cluster should it mark in a PBMC dataset?
4. A cluster has no significant marker genes after BH correction. What are three
   possible biological or technical explanations?

---
## Step 10 — Quiz

1. What does the Wilcoxon rank-sum test measure in the context of marker genes?
2. Why is the Wilcoxon test preferred over the t-test for scRNA-seq?
3. What is the one-vs-rest strategy for marker gene detection?
4. A marker gene has log2FC = 3.0 but adjusted p = 0.2. Is it a reliable marker?
5. In scanpy, what function finds marker genes and where are results stored?

---
## Step 12 — Reflection

> *[In one paragraph: explain why marker gene analysis alone is insufficient
> for cell type annotation, and describe what additional information a researcher
> typically uses to assign confident biological labels to clusters.]*

---
*Next: `08_trajectory_pseudotime_survey.ipynb`*